# Centroid candidate analysis

Reads the per-expansion JSONLs produced by `scripts/centroid_candidate_analysis.py`
and builds paper-ready figures of query length → candidate-set size by
expansion and `nprobe`.

**Inputs** (auto-discovered):
`outputs/centroid_analysis/<model>__<corpus>/<expansion>.jsonl`

**Outputs**:
Vector PDFs + 300 DPI PNGs under `outputs/centroid_analysis/<key>/figures/`.

Edit the **Style** cell to tweak palette/fonts and the **Bins** cell
to change query-length binning. Each plot cell is self-contained.

## Setup

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages

In [ ]:
# Adjust if you have multiple model+corpus keys.
REPO_ROOT = Path("..").resolve()
ANALYSIS_DIR = (
    REPO_ROOT
    / "outputs"
    / "centroid_analysis"
    / "lightonai_GTE-ModernColBERT-v1__Tevatron_webshaper-fineweb-1m-corpus"
)
CACHE_DIR = (
    REPO_ROOT
    / "outputs"
    / "centroid_analysis_cache"
    / ANALYSIS_DIR.name
)
FIG_DIR = ANALYSIS_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("analysis dir :", ANALYSIS_DIR)
print("cache dir    :", CACHE_DIR)
print("figures dir  :", FIG_DIR)

In [ ]:
# Paper-quality matplotlib defaults. Edit freely.
mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["DejaVu Serif", "Times New Roman", "Times"],
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 8.5,
    "axes.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.linewidth": 0.4,
    "grid.alpha": 0.4,
    "lines.linewidth": 1.6,
    "lines.markersize": 4,
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
    "pdf.fonttype": 42,   # editable text in PDFs
    "ps.fonttype": 42,
})

# Stable colour map for expansions; falls through to tab10 for unknown slugs.
EXPANSION_COLOR = {
    "plain":                 "#2E5EAA",
    "query_plus_reasoning":  "#C03A2B",
    "rm3_positive_t10_d1_w50_fpos": "#1F8A70",
    "rm3_positive_t20_d1_w50_fpos": "#56B870",
    "rm3_positive_t10_d1_w70_fpos": "#7E4FA0",
    "rm3_positive_t20_d1_w70_fpos": "#B188D2",
}
EXPANSION_LABEL = {
    "plain": "plain",
    "query_plus_reasoning": "+reasoning",
    "rm3_positive_t10_d1_w50_fpos": "RM3 t=10 w=0.5",
    "rm3_positive_t20_d1_w50_fpos": "RM3 t=20 w=0.5",
    "rm3_positive_t10_d1_w70_fpos": "RM3 t=10 w=0.7",
    "rm3_positive_t20_d1_w70_fpos": "RM3 t=20 w=0.7",
}

def style_for(slug: str, idx: int) -> dict:
    color = EXPANSION_COLOR.get(slug)
    if color is None:
        color = mpl.colormaps["tab10"](idx % 10)
    return {"color": color}

def label_for(slug: str) -> str:
    return EXPANSION_LABEL.get(slug, slug)

## Load data

In [ ]:
if (CACHE_DIR / "meta.json").exists():
    with (CACHE_DIR / "meta.json").open() as f:
        cache_meta = json.load(f)
    N_DOCS = int(cache_meta["n_docs"])
    K_CENTROIDS = int(cache_meta["k"])
else:
    cache_meta = {}
    N_DOCS = None
    K_CENTROIDS = None
print(f"corpus n_docs={N_DOCS!r}  k={K_CENTROIDS!r}")

rows = []
for path in sorted(ANALYSIS_DIR.glob("*.jsonl")):
    with path.open() as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            r = json.loads(line)
            r.setdefault("expansion", path.stem)
            rows.append(r)
df = pd.DataFrame(rows)
print(f"loaded rows={len(df):,}  expansions={df['expansion'].nunique()}")
df.head()

In [ ]:
if N_DOCS:
    df["pct_corpus"] = df["n_unique_docs"] / N_DOCS * 100
summary = (
    df.groupby(["expansion", "num_probes"])["n_unique_docs"]
    .agg([
        ("n", "size"),
        ("mean", "mean"),
        ("median", "median"),
        ("p95", lambda v: np.percentile(v, 95)),
        ("max", "max"),
    ])
    .round(0)
)
summary["%corpus"] = (summary["mean"] / N_DOCS * 100).round(2) if N_DOCS else None
summary

In [ ]:
qlen_stats = (
    df.drop_duplicates(["expansion", "id"])
    .groupby("expansion")["query_len"]
    .agg([
        ("n", "size"),
        ("min", "min"),
        ("median", "median"),
        ("mean", "mean"),
        ("p95", lambda v: np.percentile(v, 95)),
        ("max", "max"),
    ])
)
qlen_stats

## Length bins

Linear-scale binning. Edit `BIN_EDGES` directly, or set `AUTO_BINS=True`
and adjust `N_BINS`.

In [ ]:
AUTO_BINS = True
N_BINS = 12
MIN_PER_BIN = 30   # drop bins with fewer queries than this

if AUTO_BINS:
    qmax = int(df["query_len"].quantile(0.99))
    qmin = int(df["query_len"].min())
    BIN_EDGES = np.linspace(qmin, qmax + 1, N_BINS + 1)
else:
    BIN_EDGES = np.array([1, 5, 8, 11, 14, 17, 20, 30, 50, 100])
print("bin edges:", BIN_EDGES)

def bin_summary(sub: pd.DataFrame, value: str) -> pd.DataFrame:
    """Median + IQR per length bin (centers, drop sparse bins)."""
    idx = np.digitize(sub["query_len"], BIN_EDGES, right=False) - 1
    out = []
    for b in range(len(BIN_EDGES) - 1):
        mask = idx == b
        n = int(mask.sum())
        if n < MIN_PER_BIN:
            continue
        v = sub[value].to_numpy()[mask]
        out.append({
            "center": 0.5 * (BIN_EDGES[b] + BIN_EDGES[b + 1]),
            "median": float(np.median(v)),
            "p25": float(np.percentile(v, 25)),
            "p75": float(np.percentile(v, 75)),
            "n": n,
        })
    return pd.DataFrame(out)

## Plot 1 — candidates vs query length, overlay expansions (one panel per nprobe)

Linear x-axis. Solid line = median, shaded band = IQR (p25–p75).

In [ ]:
NPROBES_TO_PLOT = sorted(df["num_probes"].unique())
expansions = sorted(df["expansion"].unique())

n_panels = len(NPROBES_TO_PLOT)
ncols = min(3, n_panels)
nrows = (n_panels + ncols - 1) // ncols

fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(3.4 * ncols, 2.6 * nrows),
    sharex=True,
)
axes = np.atleast_2d(axes).reshape(nrows, ncols)

for i, npb in enumerate(NPROBES_TO_PLOT):
    ax = axes[i // ncols, i % ncols]
    sub_npb = df[df["num_probes"] == npb]
    for j, exp in enumerate(expansions):
        sub = sub_npb[sub_npb["expansion"] == exp]
        if sub.empty:
            continue
        bs = bin_summary(sub, "n_unique_docs")
        if bs.empty:
            continue
        st = style_for(exp, j)
        ax.fill_between(bs["center"], bs["p25"], bs["p75"], color=st["color"], alpha=0.15, lw=0)
        ax.plot(bs["center"], bs["median"], marker="o", label=label_for(exp), **st)
    ax.set_title(f"nprobe = {npb}")
    ax.set_xlabel("query length (real tokens)")
    ax.set_ylabel("unique candidate docs")
    if N_DOCS:
        ax.axhline(N_DOCS, color="gray", lw=0.6, ls=":")
    ax.set_xlim(BIN_EDGES[0], BIN_EDGES[-1])

# Hide unused axes
for k in range(n_panels, nrows * ncols):
    axes[k // ncols, k % ncols].axis("off")

# One shared legend at the bottom
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=min(len(labels), 4),
           bbox_to_anchor=(0.5, -0.02), frameon=False)
fig.tight_layout(rect=(0, 0.04, 1, 1))

fig.savefig(FIG_DIR / "candidates_vs_qlen_by_nprobe.pdf")
fig.savefig(FIG_DIR / "candidates_vs_qlen_by_nprobe.png")
plt.show()

## Plot 2 — mean candidate-set size vs nprobe, by expansion

Single-panel summary across the nprobe sweep. Y-axis log because nprobe spans 32x.

In [ ]:
fig, ax = plt.subplots(figsize=(5.4, 3.6))
for j, exp in enumerate(expansions):
    sub = df[df["expansion"] == exp]
    agg = sub.groupby("num_probes")["n_unique_docs"].agg(["mean", "median"]).sort_index()
    st = style_for(exp, j)
    ax.plot(agg.index, agg["mean"], marker="o", label=label_for(exp), **st)
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_xlabel("nprobe")
ax.set_ylabel("mean unique candidate docs")
if N_DOCS:
    ax.axhline(N_DOCS, color="gray", lw=0.6, ls=":", label=f"corpus = {N_DOCS:,}")
ax.legend(frameon=False)
ax.set_xticks(sorted(df["num_probes"].unique()))
ax.get_xaxis().set_major_formatter(mpl.ticker.ScalarFormatter())
fig.tight_layout()
fig.savefig(FIG_DIR / "candidates_vs_nprobe_by_expansion.pdf")
fig.savefig(FIG_DIR / "candidates_vs_nprobe_by_expansion.png")
plt.show()

## Plot 3 — tail distribution (CCDF) of candidate set size

P(unique docs ≥ x) per expansion at a chosen nprobe.

In [ ]:
CCDF_NPROBE = 8   # change as needed
fig, ax = plt.subplots(figsize=(5.4, 3.6))
for j, exp in enumerate(expansions):
    sub = df[(df["expansion"] == exp) & (df["num_probes"] == CCDF_NPROBE)]
    if sub.empty:
        continue
    arr = np.sort(sub["n_unique_docs"].to_numpy())
    ccdf = 1.0 - np.arange(arr.size) / arr.size
    st = style_for(exp, j)
    ax.plot(arr, ccdf, label=label_for(exp), **st)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("unique candidate docs")
ax.set_ylabel(r"$P(\geq x)$")
ax.set_title(f"nprobe = {CCDF_NPROBE}")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIG_DIR / f"candidate_ccdf_np{CCDF_NPROBE}.pdf")
fig.savefig(FIG_DIR / f"candidate_ccdf_np{CCDF_NPROBE}.png")
plt.show()

## Plot 4 — candidates vs query length, faceted by expansion (overlay nprobes)

Per-expansion view, with all nprobe lines on one panel each. Helpful when
expansion shifts the overall curve in shape, not just height.

In [ ]:
n_exp = len(expansions)
ncols = min(3, n_exp)
nrows = (n_exp + ncols - 1) // ncols

fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(3.4 * ncols, 2.6 * nrows),
    sharex=True, sharey=True,
)
axes = np.atleast_2d(axes).reshape(nrows, ncols)
cmap = mpl.colormaps["viridis"]
nprobes_sorted = sorted(df["num_probes"].unique())
np_colors = {npb: cmap(i / max(1, len(nprobes_sorted) - 1)) for i, npb in enumerate(nprobes_sorted)}

for i, exp in enumerate(expansions):
    ax = axes[i // ncols, i % ncols]
    sub_e = df[df["expansion"] == exp]
    for npb in nprobes_sorted:
        sub = sub_e[sub_e["num_probes"] == npb]
        bs = bin_summary(sub, "n_unique_docs")
        if bs.empty:
            continue
        ax.plot(bs["center"], bs["median"], color=np_colors[npb], marker="o",
                label=f"nprobe={npb}")
    ax.set_title(label_for(exp))
    ax.set_xlabel("query length (real tokens)")
    ax.set_ylabel("unique candidate docs")
    ax.set_yscale("log")

for k in range(n_exp, nrows * ncols):
    axes[k // ncols, k % ncols].axis("off")

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=len(nprobes_sorted),
           bbox_to_anchor=(0.5, -0.02), frameon=False)
fig.tight_layout(rect=(0, 0.04, 1, 1))
fig.savefig(FIG_DIR / "candidates_vs_qlen_by_expansion.pdf")
fig.savefig(FIG_DIR / "candidates_vs_qlen_by_expansion.png")
plt.show()

## Saved figures

In [ ]:
for p in sorted(FIG_DIR.iterdir()):
    print(p.relative_to(REPO_ROOT))